In [13]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

# Built-in Tool - DuckDuckGo Search

In [14]:
!pip install -U ddgs

In [15]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool=DuckDuckGoSearchRun()
results=search_tool.invoke("top news in world")
print(results)

Stay informed with top world news today. The Associated Press aims to keep you up-to-date with breaking world news stories around the globe. Get all the latest news, live updates and content about the World from across the BBC. Latest World news news, comment and analysis from the Guardian, the world's leading liberal voice. RT delivers latest news on current events from around the world including special reports, viral news and exclusive videos. With 1,700 journalists reporting from more than 150 countries, we provide live updates, investigations, photos and video of international and regional news, politics, business, technology, science, health, arts, sports and opinion.


In [16]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


# built in tool : shell tool


In [17]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('dir')

print(results)

Executing command:
 dir
sample_data



/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [18]:
shell_tool.description

'Run shell commands on this Linux machine.'

# custom tools

In [19]:
from langchain_core.tools import tool


In [20]:
# step 1---create function
def multiply(a,b):
  """multiply two numbers"""
  return a*b

In [21]:
# step 2 -- add type hints
def multiply(a:int,b:int) -> int:
  """multiply two numbers"""
  return a*b

In [22]:
# step 3 ---add tool decorator to make it tool
@tool
def multiply(a:int,b:int) -> int:
  """multiply two numbers"""
  return a*b


In [24]:
multiply.invoke({"a":4,"b":8})

32

In [25]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [26]:
print(multiply.args_schema.model_json_schema()) # the langchain recieve this json format of the tool

{'description': 'multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


# method 2--using structuredTool

In [28]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel,Field

In [29]:
class multiplyinput(BaseModel):
  a: int=Field(required=True, description="the first number to add")
  b: int=Field(required=True, description="the second number to add")


/tmp/ipykernel_1963/2094925839.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int=Field(required=True, description="the first number to add")
/tmp/ipykernel_1963/2094925839.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int=Field(required=True, description="the second number to add")


In [30]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [32]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=multiplyinput
)

In [33]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'the first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'the second number to add', 'title': 'B', 'type': 'integer'}}


In [34]:
# method 3 --using basetool

In [35]:
from langchain_core.tools import BaseTool
from typing import Type

In [36]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_1963/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_1963/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [37]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [38]:
multiply_tool = MultiplyTool()

In [39]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


# toolkit

In [40]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [41]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [42]:

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
